# 01 — Download the sector-allocation market dataset

This notebook has one responsibility: download and save the daily market dataset used by the sector-allocation experiment.

- **Tradable universe:** all nine original Select Sector SPDR ETFs: XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, and XLY
- **Context only:** SPY for the broad U.S. equity market and the VIX index for the volatility regime
- **History:** January 1999 through the latest available trading day
- **Prices:** split- and dividend-adjusted OHLC (`auto_adjust=True`)

All file paths are explicit and relative to the `notebooks/` directory, so run this notebook with `notebooks/` as the working directory.

In [1]:
from pathlib import Path

import pandas as pd
import yfinance as yf

## Configuration


In [2]:
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "1999-01-01"
END_DATE = "2026-10-01"
INTERVAL = "1d"
AUTO_ADJUST = True  # Automatically adjust prices for splits and dividends
MIN_TRADABLE_HISTORY_YEARS = 25

TRADABLE_SECTOR_ETFS = {
    "XLB": ("Materials", "Materials Select Sector SPDR Fund"),
    "XLE": ("Energy", "Energy Select Sector SPDR Fund"),
    "XLF": ("Financials", "Financial Select Sector SPDR Fund"),
    "XLI": ("Industrials", "Industrial Select Sector SPDR Fund"),
    "XLK": ("Information Technology", "Technology Select Sector SPDR Fund"),
    "XLP": ("Consumer Staples", "Consumer Staples Select Sector SPDR Fund"),
    "XLU": ("Utilities", "Utilities Select Sector SPDR Fund"),
    "XLV": ("Health Care", "Health Care Select Sector SPDR Fund"),
    "XLY": ("Consumer Discretionary", "Consumer Discretionary Select Sector SPDR Fund"),
}

CONTEXT_SERIES = {
    "SPY": "S&P 500 ETF",
    "^VIX": "CBOE Volatility Index",
}

print(f"Raw data directory: {RAW_DIR}")

Raw data directory: ../data/raw


## Build the requested symbol set

In [3]:
tradable_symbols = list(TRADABLE_SECTOR_ETFS)
context_symbols = list(CONTEXT_SERIES)
download_symbols = tradable_symbols + context_symbols

symbol_metadata = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "name": label,
            "sector": sector,
            "tradable": True,
        }
        for symbol, (sector, label) in TRADABLE_SECTOR_ETFS.items()
    ]
    + [
        {
            "symbol": symbol,
            "name": label,
            "sector": pd.NA,
            "tradable": False,
        }
        for symbol, label in CONTEXT_SERIES.items()
    ]
)

print(f"Tradable assets: {len(tradable_symbols)}")
print(f"Context series:  {len(context_symbols)}")
display(symbol_metadata)

Tradable assets: 9
Context series:  2


,symbol,name,sector,tradable
0,XLB,Materials Select Sector SPDR Fund,Materials,True
1,XLE,Energy Select Sector SPDR Fund,Energy,True
2,XLF,Financial Select Sector SPDR Fund,Financials,True
3,XLI,Industrial Select Sector SPDR Fund,Industrials,True
4,XLK,Technology Select Sector SPDR Fund,Information Technology,True
5,XLP,Consumer Staples Select Sector SPDR Fund,Consumer Staples,True
6,XLU,Utilities Select Sector SPDR Fund,Utilities,True
7,XLV,Health Care Select Sector SPDR Fund,Health Care,True
8,XLY,Consumer Discretionary Select Sector SPDR Fund,Consumer Discretionary,True
9,SPY,S&P 500 ETF,<NA>,False


## Download daily data from Yahoo Finance

Yahoo's end date is exclusive when one is supplied. `END_DATE` is left unset so the notebook retrieves the latest available data.

In [4]:
raw_wide = yf.download(
    tickers=download_symbols,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=AUTO_ADJUST,
    actions=True,
    group_by="ticker",
    threads=True,
    progress=True,
    keepna=True,
)

raw_wide = raw_wide.sort_index()
raw_wide.index = pd.to_datetime(raw_wide.index).tz_localize(None)
raw_wide.tail()

[*********************100%***********************]  11 of 11 completed


Ticker            XLU                                                         \
Price            Open       High        Low      Close      Volume Dividends   
Date                                                                           
2026-09-24  39.880001  39.889999  39.360001  39.360001  25586500.0       0.0   
2026-09-25  39.410000  39.599998  39.130001  39.509998  30294900.0       0.0   
2026-09-28  39.490002  39.549999  39.060001  39.250000  33073600.0       0.0   
2026-09-29  39.160000  39.799999  39.029999  39.709999  49172600.0       0.0   
2026-09-30  39.830002  39.939999  39.419998  39.439999  55592900.0       0.0   

Ticker                                  ^VIX             ...          XLB  \
Price      Stock Splits Capital Gains   Open       High  ... Stock Splits   
Date                                                     ...                
2026-09-24          0.0           0.0  15.83  16.570000  ...          0.0   
2026-09-25          0.0           0.0  15.61  15.940000  ...          0.0   
2026-09-28          0.0           0.0  16.16  16.620001  ...          0.0   
2026-09-29          0.0           0.0  16.17  16.440001  ...          0.0   
2026-09-30          0.0           0.0  15.93  16.610001  ...          0.0   

Ticker                           XLI                                      \
Price      Capital Gains        Open        High         Low       Close   
Date                                                                       
2026-09-24           0.0  168.750000  170.050003  168.250000  168.830002   
2026-09-25           0.0  169.529999  171.029999  169.029999  170.429993   
2026-09-28           0.0  169.360001  169.940002  167.940002  168.779999   
2026-09-29           0.0  169.119995  170.479996  168.250000  169.130005   
2026-09-30           0.0  169.570007  169.789993  166.929993  166.979996   

Ticker                                                      
Price          Volume Dividends Stock Splits Capital Gains  
Date                                                        
2026-09-24  8598900.0       0.0          0.0           0.0  
2026-09-25  7292700.0       0.0          0.0           0.0  
2026-09-28  7822800.0       0.0          0.0           0.0  
2026-09-29  6599900.0       0.0          0.0           0.0  
2026-09-30  7542800.0       0.0          0.0           0.0  

[5 rows x 87 columns]

## Normalize and validate


In [5]:
frames = []
for symbol in download_symbols:
    frame = raw_wide[symbol].copy()
    frame.columns.name = None
    frame.index.name = "date"
    frame = frame.reset_index()
    frame.insert(1, "symbol", symbol)
    frame.columns = [str(column).lower().replace(" ", "_") for column in frame.columns]
    frames.append(frame)

market_data = pd.concat(frames, ignore_index=True)
market_data = market_data.merge(
    symbol_metadata,
    on="symbol",
    how="left",
)

In [6]:
market_data = (
    market_data.loc[
        market_data["close"].notna(),
        [
            "date",
            "symbol",
            "name",
            "sector",
            "tradable",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "dividends",
            "stock_splits",
        ],
    ]
    .sort_values(["date", "symbol"])
    .reset_index(drop=True)
)

In [7]:
tradable_history = (
    market_data.loc[
        market_data["tradable"] & market_data["close"].notna()
    ]
    .groupby("symbol")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observations=("close", "count"),
    )
    .reset_index()
)

tradable_history["history_years"] = (
    tradable_history["last_date"] - tradable_history["first_date"]
).dt.days / 365.25

display(tradable_history)

assert (
    tradable_history["history_years"] >= MIN_TRADABLE_HISTORY_YEARS
).all(), "At least one sector ETF has insufficient history."

,symbol,first_date,last_date,observations,history_years
0,XLB,1999-01-04,2026-09-30,6978,27.737166
1,XLE,1999-01-04,2026-09-30,6978,27.737166
2,XLF,1999-01-04,2026-09-30,6978,27.737166
3,XLI,1999-01-04,2026-09-30,6978,27.737166
4,XLK,1999-01-04,2026-09-30,6978,27.737166
5,XLP,1999-01-04,2026-09-30,6978,27.737166
6,XLU,1999-01-04,2026-09-30,6978,27.737166
7,XLV,1999-01-04,2026-09-30,6978,27.737166
8,XLY,1999-01-04,2026-09-30,6978,27.737166


## Save the data


In [8]:
data_path = RAW_DIR / "yahoo_daily_adjusted.csv.gz"

market_data.to_csv(
    data_path,
    index=False,
    compression="gzip",
)

print(
    f"Saved {len(market_data):,} rows for "
    f"{market_data['symbol'].nunique()} symbols."
)
print(f"Data: {data_path}")

Saved 76,760 rows for 11 symbols.
Data: ../data/raw/yahoo_daily_adjusted.csv.gz
